# Patent Topic Modeling with LDA

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
import tomotopy as tp
from collections import Counter
# specify which corpus version to load (used to build the file path below)
version_prefix = "v5"

## Data Loading and Preprocessing

In [7]:


# load data for the chosen prefix
base_data = Path("../../data/claims_added")
data_path = base_data / f"{version_prefix}_processed.csv"

df = pd.read_csv(data_path)

print(f"Dataset loaded: {df.shape[0]} patents")
df["text"] = df["claims"]  # downstream code uses this name

# Quick tensor check
tensor_count = df["text"].str.contains("tensor", case=False, na=False).sum()
print(f"Patents containing 'tensor': {tensor_count}")

# load custom stopwords from file
stopfile = Path("custom_stopwords.txt")
if not stopfile.exists():
    raise FileNotFoundError(f"custom stopwords file not found: {stopfile}")

with open(stopfile, "r") as f:
    custom_stopwords = [line.strip() for line in f if line.strip()]

# combine English stopwords with custom stopwords
stop_words = set(ENGLISH_STOP_WORDS)
for w in custom_stopwords:
    for part in w.split():
        stop_words.add(part.lower())

# text cleaning utility
def clean_text(s: str) -> str:
    s = str(s).lower()
    s = re.sub(r"[^a-z0-9_\s\-]", " ", s)   # keep alnum/_/-
    s = re.sub(r"\b\d{1,2}\b", " ", s)      # remove standalone 1-2 digit numbers
    s = re.sub(r"\s+", " ", s).strip()
    return s

def tokenize_unigrams(s: str) -> list[str]:
    s = clean_text(s)
    tokens = re.findall(r"\b[a-zA-Z][a-zA-Z0-9_]{1,}\b", s)
    tokens = [t for t in tokens if t not in stop_words]
    return tokens

def build_bigram_vocab(token_lists: list[list[str]], min_count: int = 100) -> set[tuple[str, str]]:
    """
    Build a vocabulary of frequent adjacent bigrams across the corpus.
    """
    bigram_counts = Counter()

    for tokens in token_lists:
        for i in range(len(tokens) - 1):
            bigram_counts[(tokens[i], tokens[i + 1])] += 1

    return {bg for bg, count in bigram_counts.items() if count >= min_count}

def add_bigrams(tokens: list[str], bigram_vocab: set[tuple[str, str]]) -> list[str]:
    """
    Keep unigrams and also append frequent bigrams as underscore-joined tokens.
    Example:
      ['tensor', 'core', 'unit'] ->
      ['tensor', 'core', 'unit', 'tensor_core', 'core_unit']  # if both are in vocab
    """
    out = list(tokens)

    for i in range(len(tokens) - 1):
        bg = (tokens[i], tokens[i + 1])
        if bg in bigram_vocab:
            out.append(tokens[i] + "_" + tokens[i + 1])

    return out

# Step 1: cleaned text
df["text_clean"] = df["text"].map(clean_text)

# Step 2: unigram tokenization
df["tokens_unigram"] = df["text"].map(tokenize_unigrams)

# drop docs that are already empty
df = df[df["tokens_unigram"].map(len) > 0].copy()

# Step 3: build corpus-wide bigram vocab
bigram_vocab = build_bigram_vocab(df["tokens_unigram"].tolist(), min_count=100)

print(f"Frequent bigrams kept: {len(bigram_vocab)}")
print("Sample bigrams:", [f"{a}_{b}" for a, b in list(sorted(bigram_vocab))[:20]])

# Step 4: add bigrams into each document
df["tokens"] = df["tokens_unigram"].map(lambda toks: add_bigrams(toks, bigram_vocab))

# final drop of any empty docs, just in case
df = df[df["tokens"].map(len) > 0].copy()

print(f"Usable documents: {len(df)}")
print(f"Average unigram tokens per doc: {df['tokens_unigram'].map(len).mean():.1f}")
print(f"Average tokens per doc after bigrams: {df['tokens'].map(len).mean():.1f}")

# quick sanity check
for i in range(min(3, len(df))):
    print(f"\nDoc {i} sample tokens:")
    print(df.iloc[i]["tokens"][:40])

Dataset loaded: 31655 patents
Patents containing 'tensor': 824
Frequent bigrams kept: 20091
Sample bigrams: ['abs_abs', 'absolute_difference', 'absolute_differences', 'absolute_value', 'absolute_values', 'abstract_syntax', 'abstraction_layer', 'ac_power', 'accelerated_data', 'accelerated_processing', 'acceleration_according', 'acceleration_component', 'acceleration_data', 'acceleration_hardware', 'acceleration_structure', 'accelerator_accelerator', 'accelerator_according', 'accelerator_card', 'accelerator_chip', 'accelerator_circuit']
Usable documents: 31647
Average unigram tokens per doc: 571.8
Average tokens per doc after bigrams: 779.7

Doc 0 sample tokens:
['integrated', 'circuit', 'data', 'processing', 'circuit', 'input', 'stage', 'combinatorial', 'logic', 'stage', 'output', 'stage', 'input', 'stage', 'responsive', 'clock', 'signal', 'arranged', 'set', 'data', 'signals', 'set', 'data', 'signals', 'provide', 'set', 'data', 'signals', 'input', 'combinatorial', 'logic', 'stage', 'por

## LDA Model Fitting

In [8]:
n_topics = 20

mdl = tp.LDAModel(
    k=n_topics,
    alpha=0.3,   # like doc_topic_prior
    eta=0.01,    # like topic_word_prior
    min_df=10,   # similar spirit to CountVectorizer(min_df=10)
    rm_top=0,    # can raise later if generic patent terms dominate
    seed=42
)

for tokens in df["tokens"]:
    mdl.add_doc(tokens)

print(f"Documents in model: {len(mdl.docs)}")
print(f"Vocabulary size: {mdl.num_vocabs}")

Documents in model: 31647
Vocabulary size: 0


In [9]:
print("docs before train:", len(mdl.docs))
print("vocabs before train:", mdl.num_vocabs)

mdl.train(1)

print("docs after train:", len(mdl.docs))
print("vocabs after train:", mdl.num_vocabs)

docs before train: 31647
vocabs before train: 0
docs after train: 31647
vocabs after train: 29474


/Users/noahfehr/anaconda3/lib/python3.12/site-packages/tomotopy/models.py:637: RuntimeWarning: The training result may differ even with fixed seed if `workers` != 1.
  return self._train(iterations, workers, parallel, freeze_topics, callback_interval, callback)


In [10]:
print("Training LDA model...")
for i in range(0, 100, 10):
    mdl.train(10)
    print(f"Iteration: {i+10}\tLog-likelihood per word: {mdl.ll_per_word:.4f}")


Training LDA model...
Iteration: 10	Log-likelihood per word: -8.6770
Iteration: 20	Log-likelihood per word: -8.2140
Iteration: 30	Log-likelihood per word: -8.0830
Iteration: 40	Log-likelihood per word: -8.0223
Iteration: 50	Log-likelihood per word: -7.9908
Iteration: 60	Log-likelihood per word: -7.9734
Iteration: 70	Log-likelihood per word: -7.9621
Iteration: 80	Log-likelihood per word: -7.9545
Iteration: 90	Log-likelihood per word: -7.9488
Iteration: 100	Log-likelihood per word: -7.9443


In [11]:
doc_topic_dist = np.array([doc.get_topic_dist() for doc in mdl.docs])

print(f"LDA fitted with {n_topics} topics")
print("Avg max topic weight:", doc_topic_dist.max(axis=1).mean())
print(f"Final log-likelihood per word: {mdl.ll_per_word:.4f}")

LDA fitted with 20 topics
Avg max topic weight: 0.4336807
Final log-likelihood per word: -7.9443


## View topics

In [12]:
topic_summaries = {}

for topic_idx in range(mdl.k):
    top_words = [w for w, _ in mdl.get_topic_words(topic_idx, top_n=15)]
    topic_summaries[topic_idx] = top_words

    print(f"Topic {topic_idx:02d}: " + ", ".join(top_words))

Topic 00: image, region, images, pixel, image_data, pixels, image_processing, according, regions, processing, area, color, step, input_image, digital
Topic 01: server, key, identifier, access, request, information, communication, authentication, mobile, electronic, mac, security, message, application, using
Topic 02: instruction, vector, element, elements, register, operation, array, input, matrix, result, processor, bit, output, simd, set
Topic 03: value, values, number, bit, table, index, set, bits, point, threshold, quantization, quantized, maximum, range, error
Topic 04: data, storage, input_data, data_processing, structure, set, data_set, data_structure, data_data, stream, processing, output_data, stored, data_storage, portion
Topic 05: computing, hardware, application, resource, resources, software, component, configuration, graph, performance, execution, compute, accelerator, instance, set
Topic 06: network, node, neural, neural_network, layer, nodes, input, weight, output, laye

In [ ]:
from stability import compute_topic_stability

k_values = [16, 18, 20, 22, 24, 26, 28]
seeds = [0, 1, 2, 3, 4]

results_df = compute_topic_stability(
    tokenized_docs=df["tokens"].tolist(),
    k_values=k_values,
    seeds=seeds,
    iterations=100,
    vocab_min_df=10,
    min_df=10,
    rm_top=0
)

print("\n===== SUMMARY =====")
print(results_df.sort_values("k"))

Fixed comparison vocab size: 29474

===== k = 16 =====
